# 01 – Univariate Analysis

## Credit Risk – Probability of Default

# 1. Libs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, HTML
from IPython.display import display_html
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

__Also telling pandas to show all columns and seaborn to use a white background.__

# 2. Data Loading

Loading the Default of Credit Card Clients dataset from the UCI Machine Learning Repository.


In [ ]:
default_of_credit_card_clients = fetch_ucirepo(id=350)

X = default_of_credit_card_clients.data.features
y = default_of_credit_card_clients.data.targets

print("Predictors shape:", X.shape)
print("Target shape:", y.shape)

__We have 30,000 observations, 23 predictor variables, and 1 target variable.__

# 3. Data Preparation

Renaming the dataset columns using descriptive variable names based on the UCI documentation and combining the predictor variables with the target variable into a single DataFrame.


In [ ]:
column_mapping = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    "X6": "PAY_1",
    "X7": "PAY_2",
    "X8": "PAY_3",
    "X9": "PAY_4",
    "X10": "PAY_5",
    "X11": "PAY_6",
    "X12": "BILL_AMT1",
    "X13": "BILL_AMT2",
    "X14": "BILL_AMT3",
    "X15": "BILL_AMT4",
    "X16": "BILL_AMT5",
    "X17": "BILL_AMT6",
    "X18": "PAY_AMT1",
    "X19": "PAY_AMT2",
    "X20": "PAY_AMT3",
    "X21": "PAY_AMT4",
    "X22": "PAY_AMT5",
    "X23": "PAY_AMT6"
}

X = X.rename(columns=column_mapping)

df = pd.concat([X, y], axis=1)

target = y.columns[0]
  
df.head()

__Now we have descriptive names for our predictors.__

__For further information, please visit__ _notebooks/Data Dictionary — Credit Risk - Probability of Default.md_ 

In [ ]:
# saving the dataset

data_dir = Path("../data/processed")
data_dir.mkdir(parents=True, exist_ok=True)

df.to_parquet(data_dir / "credit_default_eda.parquet", index=False)

# 4. Initial Data Inspection

Inspecting the dataset structure, including its dimensions, column names, data types and missing values, to assess data consistency and identify potential data quality issues.
    

In [ ]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

# Create DataFrames
data_types = df.dtypes.to_frame(name="data_type")

missing_values = (
    df.isnull()
      .sum()
      .to_frame(name="missing_values")
)

# Create output widgets
left = widgets.Output()
right = widgets.Output()

# Left table
with left:
    print("Data Types")
    display(data_types)

# Right table
with right:
    print("Missing Values")
    display(missing_values)

# Display side by side
display(widgets.HBox([left, right]))

__Perfect. The inspection confirms three important points:__

1. Dimensions: 30,000 observations and 24 columns, as expected.
2. Missing values: no variable contains NaNs.
3. Data types: all columns are stored as int64.
   
__One caveat: int64 describes the storage format, not the statistical nature of the variable. SEX, EDUCATION, and MARRIAGE, for example, remain categorical.__

# 5. Target Variable Analysis

Analyzing the target variable distribution to understand the proportion of default and non-default observations and assess class imbalance.

In [ ]:
left = widgets.Output()
right = widgets.Output()

# Output da tabela
with left:
    target_distribution = pd.DataFrame({
        "count": df[target].value_counts().sort_index(),
        "percentage": df[target].value_counts(normalize=True).sort_index() * 100
    })

    display(target_distribution)

# Output do gráfico
with right:
    plt.figure(figsize=(7, 5))

    sns.countplot(data=df, x=target)

    plt.title("Distribution of Credit Default")
    plt.xlabel("Default in the following month")
    plt.ylabel("Number of clients")

    plt.show()

display(widgets.HBox([left, right]))

__This represents approximately 3.52 non-defaulting clients for every one who defaults.__

__We therefore have a moderate imbalance. While this does not automatically necessitate techniques such as SMOTE or undersampling, we will need to be careful when evaluating our models.__

__For instance, a model that always predicted 0 would achieve 77.88% accuracy, yet it would fail to identify any defaulting clients. Consequently, we cannot rely solely on accuracy to evaluate our model.__

# 6. Descriptive Statistics of Numerical Variables

Generating descriptive statistics for numerical variables, including measures of central tendency, dispersion, and range, to understand their distributions and identify potential outliers.

In [ ]:
numeric_columns = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

numeric_summary = df[numeric_columns].describe().T

display(numeric_summary)

__This table already reveals some important patterns. We should not remove any extreme values at this stage; first, we need to determine whether they are valid and how they affect the distribution.__

What the results show:
   
__LIMIT_BAL — credit limit__
- Mean: __NT 167,484__
- Median: __NT 140,000__
- Maximum: __NT 1,000,000__

The fact that the mean is higher than the median suggests a right-skewed distribution: some clients have very high credit limits.

__AGE — age__
- Mean: __35.49 years__
- Median: __34 years__
- Maximum: __79 years__

The distribution appears less skewed than the other variables, though we still need to visualize the data.

__BILL_AMT1 to BILL_AMT6 — bill amounts__

An interesting observation here: the means range from approximately __NT 38,000__ to __NT 51,000__, while the medians fall between __NT 1,200__ and __NT 3,600__.

This discrepancy suggests highly skewed distributions, with some very large values.

There are also negative values, such as BILL_AMT6 = -339,603. We should not automatically classify these as errors; they could represent credit balances or account adjustments. We need to verify their meaning before deciding how to handle them.

__PAY_AMT1 to PAY_AMT6 — payment amounts__

- The medians are relatively low, ranging from approximately __NT 252__ to __NT 1,000__.
- The means are significantly higher.
- There are very high maximum values, reaching up to __NT 1,684,259__.
- The minimum value is zero for all periods.
  
This points to a strong right skew and suggests that many clients may not have made any payments during certain periods. The analysis confirms that several variables exhibit strong right-skewness, particularly the bill amounts (BILL_AMT) and payment amounts (PAY_AMT).

This is evident from the difference between the mean and the median. For example:

- PAY_AMT2: a mean of approximately __NT 5,921__, but a median of __NT 833__.
- BILL_AMT1: a mean of approximately __NT 51,223__, but a median of __NT 22, 381__.
- LIMIT_BAL: a mean of __NT 167,484__, with a median of __NT 140,000__.
  
This indicates that some clients have values significantly higher than those of the majority.

Next, lets investigate the negative values on the __BILL_AMT*__ variables.

# 7. Detection of Negative Bill Amounts

Checking the frequency of negative values in monthly bill statement amounts to identify potentially relevant account balances or anomalies that require further investigation.

In [ ]:
bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]

negative_bill_values = (
    df[bill_columns]
    .lt(0)
    .sum()
    .to_frame(name="negative_values")
)

display(negative_bill_values)

__The results show that negative values appear in all six periods, but they represent a small portion of the dataset.__

| Variable   | Negative values | % of 30,000 customers |
|------------|-----------------|-----------------------|
| BILL_AMT1  | 590             | 1.97%                 |
| BILL_AMT2  | 669             | 2.23%                 |
| BILL_AMT3  | 655             | 2.18%                 |
| BILL_AMT4  | 675             | 2.25%                 |
| BILL_AMT5  | 655             | 2.18%                 |
| BILL_AMT6  | 688             | 2.29%                 |

__What can we conclude?__

1. These do not appear to be isolated occurrences limited to a single month.
   
2. The proportion is relatively stable, ranging between 1.97% and 2.29%.

3. We do not have sufficient evidence to classify them as errors or to remove them.

__One possible interpretation is that they represent credit balances on the card account—for instance, resulting from overpayments or adjustments. However, we need to verify this hypothesis against what the documentation actually confirms.__

__Later we will investigate this further by examining its relationship with our target variable.__

# 8. Default Rate Analysis of Negative Bill Amounts

Comparing default rates between clients with negative and non-negative monthly bill statement amounts to investigate whether negative balances are associated with different credit risk profiles.

In [ ]:
results = []

for col in bill_columns:

    negative_col = f"{col}_NEGATIVE"
    df[negative_col] = df[col] < 0

    result = (
        df.groupby(negative_col)[target]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "number_of_clients",
            "mean": "default_rate"
        })
    )

    result["default_rate"] *= 100

    results.append(
        f"<h4>{col}</h4>" + result.to_html(float_format=lambda x: f"{x:.2f}")
    )

left_column = "".join(results[:3])
right_column = "".join(results[3:])

html = f"""
<div style="display: flex; gap: 30px;">
    <div style="flex: 1;">{left_column}</div>
    <div style="flex: 1;">{right_column}</div>
</div>
"""

display_html(html, raw=True)

__We observe a consistent pattern: negative invoice amounts are associated with a lower default rate within this sample.__

__This could be consistent with the hypothesis that these customers hold credit balances in their accounts. However, it remains unclear whether this characteristic truly reflects distinct financial behavior or if it is linked to other variables.__

__We must also take into account that these groups are small, representing approximately 2% of customers in each period.__

__My decision for now: keep the negative values. We have no justification for removing or replacing them.__

__Later, let's also create a feature based on negative bill amounts that could be useful for our model later.__

# 9. Numerical Variable Distributions

Visualizing the distributions of credit limits and customer ages using histograms to examine their shapes, concentration, spread, and potential skewness or outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="LIMIT_BAL", bins=40, ax=axes[0])
axes[0].set_title("Credit Limit Distribution")

sns.histplot(data=df, x="AGE", bins=30, ax=axes[1])
axes[1].set_title("Age Distribution")

plt.tight_layout()
plt.show()

__The two histograms reveal interesting patterns:__

1. LIMIT_BAL — Credit Limit
   
- Right-skewed distribution.
- Higher concentration of customers at lower credit limits.
- Frequency decreases as the limit increases.
- There are several peaks, indicating that certain limit values ​​occur more frequently than others.
  
__This is to be expected for a credit limit variable, which typically features discrete values ​​and specific tiers for credit granting.__

2. AGE — Age
   
- Highest concentration of customers between approximately 25 and 40 years of age.
- A peak in frequency occurs around 29–30 years of age.
- Right-skewed distribution, with a tail extending up to approximately 79 years of age.

__Also, nothing evidently abnormal in these graphs.__

# 10. Monthly Bill Amount Distributions

Visualizing the distributions of monthly bill statement amounts using histograms to examine their shapes, variability, skewness, and potential outliers across the six billing periods.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, bill_columns):
    sns.histplot(data=df, x=col, bins=40, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

__Analysis of the histograms__

__Right-skewness (positive skewness)__

All six variables exhibit strong right-skewness. Most customers have relatively low bill amounts, while a smaller proportion shows very high values.

This confirms what we observed in the descriptive statistics: the means are considerably higher than the medians.

__Extreme values (outliers)__

All six charts show long right-hand tails.

The most notable case is BILL_AMT3, which has a maximum value of approximately NT$ 1.66 million. This value significantly expands the horizontal scale, visually compressing the distribution of the remaining customers.

__Negative values__

The histograms also display negative values, which are particularly noticeable in BILL_AMT1, BILL_AMT4, and BILL_AMT6.

We have already verified that these appear in approximately 2% of the observations for each period, and we have decided to retain them.

__We will keep the original values for now, note the strong skewness and extreme values, and subsequently evaluate their relationship with default.__

# 11. Monthly Payment Amount Distributions

Visualizing the distributions of monthly payment amounts using histograms to examine their shapes, variability, skewness, and potential outliers across the six billing periods.

In [ ]:
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, payment_columns):
    sns.histplot(data=df, x=col, bins=40, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

__The histograms for the six PAY_AMT variables show an even more pronounced skewness than what we observed for BILL_AMT.__

__Lets try o visualize these distributions without letting extreme values dominate the horizontal axis, we can use the 99th percentile of each variable as a limit for the visualization.__

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, payment_columns):

    upper_limit = df[col].quantile(0.99)

    sns.histplot(
        data=df[df[col] <= upper_limit],
        x=col,
        bins=40,
        ax=ax
    )

    ax.set_title(f"{col} - Up to 99th percentile")
    ax.ticklabel_format(style="plain", axis="x")

plt.tight_layout()
plt.show()

__These histograms are much more informative. They allow us to better visualize how payment amounts are distributed among customers.__

What do we observe?
   
__All six variables continue to show strong right-skewness, even after limiting the visualization to the 99th percentile.__

This indicates that the skewness is not caused solely by the top 1% of the highest values.

We also note that:

- The highest concentration of customers falls within the lower payment ranges.
  
- Frequency decreases rapidly as the payment amount increases.
  
- The pattern is quite similar across all six periods.
  
- PAY_AMT4, PAY_AMT5, and PAY_AMT6 appear to have an especially high concentration in the lowest ranges.
  
However, we cannot yet determine exactly how many customers made zero payments; that would require counting those records separately.

2. What does this mean for modeling?
   
__We have identified an important characteristic: payment amounts vary significantly among customers.__

This suggests that, in the future, we could explore derived variables, such as:

- The average payment over the last six months.
  
- The number of months with no payments.
  
- The ratio between amounts paid and amounts billed.

These variables could help represent a customer's financial behavior, though they will need to be evaluated empirically.

__For now, we will retain the original values.__

# 12. Categorical Variable Distributions

Examining the frequency distributions of categorical variables to identify existing categories, undocumented codes, and potential inconsistencies in the dataset, with particular attention to payment status across the six billing periods.

In [ ]:
categorical_columns = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_1",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

outputs = []

for col in categorical_columns:

    output = widgets.Output(
        layout=widgets.Layout(width="32%")
    )

    with output:
        print(col)

        display(
            df[col]
            .value_counts(dropna=False)
            .sort_index()
            .to_frame(name="number_of_clients")
        )

    outputs.append(output)

# Arrange outputs in rows of 3
rows = []

for i in range(0, len(outputs), 3):
    row = widgets.HBox(outputs[i:i+3])
    rows.append(row)

display(widgets.VBox(rows))

__We now have the complete distribution of the categorical variables, allowing us to conclude the univariate inspection.__

The results reveal some important points, particularly regarding the PAY_* variables.

__SEX, EDUCATION, and MARRIAGE__
   
SEX: We found only the documented codes—1 (male) and 2 (female)—with no apparent inconsistencies.

EDUCATION: We identified additional codes (0, 5, and 6), accounting for 345 observations (1.15% of the dataset).

MARRIAGE: We found 54 observations (0.18%) with a code of 0, which is not described in the documentation.

__Our approach will be to retain these observations and treat the undocumented codes as "unknown" categories. The transformation will be performed at a later stage.__

Let's also visualize the distributions of the PAY_* variables:

In [ ]:
pay_columns = [
    "PAY_1", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flat, pay_columns):

    sns.countplot(
        data=df,
        x=col,
        order=sorted(df[col].unique()),
        ax=ax
    )

    ax.set_title(f"{col} Distribution")
    ax.set_xlabel("Payment Status")
    ax.set_ylabel("Number of Clients")

plt.tight_layout()
plt.show()

__Code 0 is the most frequent across all periods, accounting for approximately 49% to 56% of the clients.__

| Variable | Clients with Code 0 | Percentage |
|----------|--------------------:|-----------:|
| PAY_1    | 14,737              | 49.12%     |
| PAY_2    | 15,730              | 52.43%     |
| PAY_3    | 15,764              | 52.55%     |
| PAY_4    | 16,455              | 54.85%     |
| PAY_5    | 16,947              | 56.49%     |
| PAY_6    | 16,286              | 54.29%     |

__Codes -1 and -2 also appear with considerable frequency.__

This underscores the importance of understanding these codes before deciding how to represent them in the model.

__A specific detail: code 1__

Observe the frequency of code 1:

| Variable | Clients with code 1 |
|----------|--------------------:|
| PAY_1    | 3,688               |
| PAY_2    | 28                  |
| PAY_3    | 4                   |
| PAY_4    | 2                   |
| PAY_5    | 0                   |
| PAY_6    | 0                   |

__This discrepancy is quite significant.__

Although positive codes are described as periods of delay, the behavior associated with code 1 varies greatly across the different months.

We should not assume this is an error, but we also cannot ignore this peculiarity. There may be a specific recording or encoding convention used in the dataset.

__Longer delays are rare__

Codes 3 through 8 appear with relatively low frequency across all periods.

This may be relevant for data preprocessing, as some categories contain very few observations.

__However, rare categories can be important for predicting default; therefore, we should not automatically group them together.__

__Atrasos mais elevados são raros__

Os códigos 3 a 8 aparecem com frequência relativamente baixa em todos os períodos.

Isso pode ser relevante para o pré-processamento, pois algumas categorias possuem pouquíssimas observações.

__Entretanto, categorias raras podem ser importantes para prever inadimplência. Portanto, não vamos agrupá-las automaticamente.__

# Key Findings

- The target variable is moderately imbalanced, with a default rate of 22.12%.
- BILL_AMT and PAY_AMT variables exhibit strong right-skewed distributions.
- Negative bill amounts occur in approximately 2% of records per month.
- EDUCATION and MARRIAGE contain undocumented category codes.
- Payment status variables are concentrated in codes -2, -1, and 0.
- Higher payment delay codes are relatively rare.
- The distribution of payment status code 1 differs substantially across periods.